# Reproducing ResNet on CIFAR-10 (Colab GPU)

Runs the full test suite (including the data-pipeline tests that need CIFAR-10) and
trains ResNet-20 / ResNet-56 on a GPU. On Colab, CIFAR downloads in seconds.

**Runtime > Change runtime type > GPU (T4) before running.**

**Measured on a free T4:** ResNet-20 ~51 min/seed, ResNet-56 ~2.5 h/seed. Results
append to Drive the moment each run finishes, so a reclaimed session costs at most the
run in flight — rerun only the missing seeds.


In [ ]:
!git clone https://github.com/AmroAbujabal/resnet-cifar-repro.git
%cd resnet-cifar-repro
!pip -q install pyyaml pytest
import torch; print('torch', torch.__version__, '| cuda', torch.cuda.is_available())

In [ ]:
# Persist results to Drive: a reclaimed Colab session loses /content, not this file.
from google.colab import drive
drive.mount('/content/drive')
import os
RESULTS = '/content/drive/MyDrive/resnet-repro/results.csv'
os.makedirs(os.path.dirname(RESULTS), exist_ok=True)
print('results ->', RESULTS)


## 1. Full test suite (proves the data pipeline, model, metric, and training loop)
The T1 data tests download CIFAR-10 here; all others run on synthetic data.

In [ ]:
!python -m pytest -q

## 2. Train ResNet-20, 3 seeds (paper Table 6: 8.75%)
~51 min per seed on a T4 (measured at 20.8 it/s), so ~2.6 h total.


In [ ]:
for s in (0, 1, 2):
    !python scripts/train.py --config configs/resnet20.yaml --seed {s} --device cuda --results {RESULTS}


## 3. Train ResNet-56, 3 seeds (paper Table 6: 6.97%)
**~2.5 h per seed on a T4** — 3x the blocks of ResNet-20, so ~7.6 h for the set.
Free Colab will not hold one session that long; run it seed-by-seed and let the
Drive-backed `results.csv` accumulate across sessions.


In [ ]:
for s in (0, 1, 2):
    !python scripts/train.py --config configs/resnet56.yaml --seed {s} --device cuda --results {RESULTS}

## 4. Results

In [ ]:
import pandas as pd
df = pd.read_csv(RESULTS)
display(df)
print(df.groupby('model')['test_error_pct'].agg(['mean', 'std', 'count']))